import sys
from pathlib import Path
# El intérprete que ejecuta ESTE notebook. Si la ruta no contiene .venv,
# el kernel es el equivocado: Kernel -> Change Kernel.
print("Intérprete:", sys.executable)
# La carpeta desde la que se ejecuta. Las rutas relativas del notebook
# se resuelven desde aquí, no desde donde está el archivo .ipynb.
print("Carpeta de trabajo:", Path.cwd())
# Las librerías del proyecto, con su versión
for lib in ["numpy", "pandas", "matplotlib", "sklearn"]:
    try:
        print(f" {lib:12}", __import__(lib).__version__)
    except ImportError:
        print(f" {lib:12} NO DISPONIBLE en este kernel")

# Problematica
Se trabaja con el conjunto de datos oficial "tb_patentes_aguas_december_2025_data.csv", que consolida las
patentes por no uso de derechos de aprovechamiento de aguas en Chile (46.467 registros, 21 variables; Tesorería
General de la República, 2025). <br>
## Pregunta de análisis: ¿cómo se distribuyen la morosidad y el saldo impago de estas
patentes según Tesorería Regional, tipo de titular (persona natural o jurídica) y año de emisión? 
<br>
## Objetivo metodológico: 
que el análisis sea trazable y reproducible mediante Git/GitHub y Jupyter Notebooks.

## Subtítulo
### Sección
Texto normal con **negrita**, *cursiva* y `código en línea`.
Probando cambios



### Herramientas del ecosistema científico

| Pieza | Por que se usa | 
|---|---|
| Entorno virtual (`.venv`) | Aislar las versiones de las librerías del proyecto. | 
| JupyterLab + `ipykernel` | Ejecutar el cuaderno con el intérprete del proyecto. | 
| `requirements.txt` | Declarar las dependencias exactas. | 
| Git | Registrar la historia del trabajo y su autoría. | 
| GitHub | Colaborar y publicar la evidencia evaluada. | 
| `pathlib` | Rutas relativas independientes del sistema operativo. | 

En esta fase se usan sobre todo módulos de la biblioteca estándar: el cuaderno construye y
verifica el entorno, todavía no analiza datos. `numpy` y `pandas` aparecen únicamente para
comprobar que están disponibles y para escribir las tablas de documentación.

In [ ]:
import sys                        # intérprete en uso: es lo que se verifica más abajo
import json                       # exportación de metadatos legibles
import platform                   # sistema operativo, para dejarlo en la bitácora
import subprocess                 # consultas a Git desde el cuaderno
import shutil                     # localizar ejecutables (git) sin suponer que existen
import importlib                  # importar el módulo que este cuaderno va a escribir
import importlib.util             # comprobar si un paquete está instalado
from pathlib import Path          # manejo de rutas independiente del sistema operativo
from datetime import date

# Verificación del entorno: instala en el intérprete del kernel lo que falte.
print("Intérprete del kernel:", sys.executable)

REQUERIDOS = {"numpy": "numpy", "pandas": "pandas"}   # módulo: paquete pip
faltantes = [pkg for mod, pkg in REQUERIDOS.items()
             if importlib.util.find_spec(mod) is None]

if faltantes:
    print("Instalando:", ", ".join(faltantes))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *faltantes])
    importlib.invalidate_caches()

import numpy as np                # presente para verificar el entorno del proyecto
import pandas as pd               # tablas de documentación de la fase

print("numpy", np.__version__, "· pandas", pd.__version__)

# Reproducibilidad: la misma semilla que usa el cuaderno de la Fase 2.
SEMILLA = 42
np.random.seed(SEMILLA)

print("Cuaderno de la Fase 1 ·", date.today().isoformat())

## 1. Definición del problema

Deja el problema escrito como una **estructura de datos**

In [ ]:
PROYECTO = {
    "titulo": "consolidacion de patentes por no uso de derechos de aprovechamiento de aguas",
    "grupo": "Grupo 9",
    "asignatura": "MCDI500 · Programación para la Ciencia de Datos",
    "integrantes": ["Christian Abarca Araya", "Vicente Agosin Orellana", "Patricia Campos Tapia","Daniel Valdivia Tapia"],
    "problematica": (
        

        
        "Análisis de la morosidad en las patentes por no uso de derechos de aprovechamiento de aguas en Chile, a partir del "
        "conjunto de datos oficial publicado en el Portal de Datos Abiertos del Gobierno de Chile. El objetivo es describir cómo "
        "se distribuyen la morosidad y el saldo impago según Tesorería Regional, tipo de titular (persona natural o jurídica) y "
        "año de emisión, con un flujo de trabajo trazable y reproducible. "
    ),
    "objetivo_general": (
        "Construir un flujo reproducible de análisis que, a partir de datos de morosidad en las patentes por no aprovechamiento de agua "
        "en Chile, entender la distribución de morosidad segun region y tipo de titular"
    ),
    "objetivos_especificos": [
        "Definir el problema y establecer el entorno reproducible del proyecto (F1).",
        "Construir el pipeline de obtención, limpieza y transformación de datos (F2).",
        "Implementar el núcleo algorítmico con programación estructurada y POO (F3).",
        "Comunicar los hallazgos mediante visualización y un informe técnico (F4).",
    ],
    "preguntas": [
        "En qué región se concentran las mayores morosidades por no aprovechamiento de aguas",
        "Qué tipo de titular concentra mayor morosidad por no aprovechamiento de aguas",
    ],
    "criterios_exito": [
        "El repositorio se clona y el cuaderno corre completo sin intervención manual.",
        "Cada decisión de preprocesamiento queda justificada con cifras, no con adjetivos.",
        "Los cuatro integrantes tienen commits propios en el historial.",
    ],
    "alcance": {
        "incluye": ["Definición del problema", "Entorno reproducible", "Selección del conjunto"],
        "excluye": ["Limpieza e imputación", "Modelado predictivo", "Despliegue"],
    },
    "limitaciones": [
        "Los datos se asumen como verdaderos: No se comprueba su veracidad",
        "La variable objetivo está desbalanceada, lo que condiciona la evaluación posterior.",
        "El conjunto no incluye variables temporales, por lo que no hay análisis de evolución.",
    ],
}

In [ ]:
def presentar_proyecto(config):
    """
    Imprime la definicion del proyecto de forma legible.

    Parametros
    ----------
    config : dict
        Diccionario de configuracion del proyecto.

    Retorna
    -------
    None

    Lanza
    -----
    KeyError
        Si falta alguna de las claves obligatorias.
    """
    obligatorias = ("titulo", "problematica", "objetivo_general", "objetivos_especificos")
    faltantes = [c for c in obligatorias if c not in config]
    if faltantes:
        # Se falla temprano y con un mensaje que dice QUE falta, no solo que algo falló.
        raise KeyError(f"Faltan claves obligatorias en la configuracion: {faltantes}")

    print(config["titulo"].upper())
    print("=" * 60)
    print(f"{config['asignatura']} · {config['grupo']}")
    print("\nProblemática")
    print(config["problematica"])
    print("\nObjetivo general")
    print(config["objetivo_general"])
    print("\nObjetivos específicos")
    for i, obj in enumerate(config["objetivos_especificos"], start=1):
        print(f"  {i}. {obj}")
    print("\nPreguntas que orientan el trabajo")
    for pregunta in config["preguntas"]:
        print(f"  · {pregunta}")


presentar_proyecto(PROYECTO)

In [ ]:
print("Alcance de la Fase 1")
print("  Incluye:", ", ".join(PROYECTO["alcance"]["incluye"]))
print("  Excluye:", ", ".join(PROYECTO["alcance"]["excluye"]))

print("\nLimitaciones declaradas")
for lim in PROYECTO["limitaciones"]:
    print(f"  · {lim}")

## 2. Verificación del entorno

**Qué hace este paso.** Comprueba, con evidencia y no por confianza, que el cuaderno se
está ejecutando con el intérprete del proyecto y que las librerías declaradas están
disponibles en él.

In [ ]:
DEPENDENCIAS = ["numpy", "pandas", "matplotlib", "sklearn"]

# Correspondencia entre el nombre de importación y el nombre del paquete en PyPI:
# se instala scikit-learn, pero se importa sklearn.
NOMBRE_EN_PYPI = {"sklearn": "scikit-learn"}


def verificar_entorno(dependencias):
    """
    Comprueba interprete, entorno virtual, carpeta de trabajo y librerias.

    Parametros
    ----------
    dependencias : list of str
        Nombres de importacion de las librerias que el proyecto declara.

    Retorna
    -------
    dict
        Resultado de cada comprobacion, para dejarlo en la bitacora de la fase.
    """
    reporte = {}

    # 1. Intérprete que ejecuta ESTE cuaderno. Si la ruta no contiene .venv,
    #    el kernel no es el del proyecto: Kernel -> Change Kernel.
    ejecutable = Path(sys.executable)
    en_venv = ".venv" in ejecutable.parts or sys.prefix != sys.base_prefix
    reporte["interprete"] = str(ejecutable)
    reporte["entorno_virtual"] = bool(en_venv)
    print("Intérprete       :", ejecutable)
    print("Entorno virtual  :", "[OK] activo" if en_venv else "[AVISO] parece el Python del sistema")

    # 2. Carpeta de trabajo: las rutas relativas se resuelven desde aquí,
    #    no desde donde está guardado el archivo .ipynb.
    reporte["carpeta_trabajo"] = str(Path.cwd())
    print("Carpeta de trabajo:", Path.cwd())
    print("Sistema          :", platform.system(), platform.release())
    print("Python           :", sys.version.split()[0])

    # 3. Librerías del proyecto, con su versión.
    print("\nLibrerías declaradas")
    versiones = {}
    for nombre in dependencias:
        try:
            modulo = importlib.import_module(nombre)
            version = getattr(modulo, "__version__", "sin atributo __version__")
            versiones[nombre] = version
            print(f"  [OK]    {nombre:12} {version}")
        except ImportError:
            # No se interrumpe el cuaderno: se informa qué instalar y con qué nombre.
            versiones[nombre] = None
            paquete = NOMBRE_EN_PYPI.get(nombre, nombre)
            print(f"  [FALTA] {nombre:12} instale con: python -m pip install {paquete}")
    reporte["versiones"] = versiones
    return reporte


ENTORNO = verificar_entorno(DEPENDENCIAS)

## 3. Estructura del repositorio y artefactos

**Qué hace este paso.** Crea las carpetas del proyecto y escribe desde el cuaderno los dos
archivos que hacen reproducible el entorno: `.gitignore` y `requirements.txt`.

**Por qué desde el cuaderno.** Porque así quedan derivados de la configuración declarada en
la sección 1 y no de lo que alguien recordó escribir a mano. Es la misma lógica de la Fase
2: el artefacto se genera, se verifica y queda trazado.

In [ ]:
def localizar_raiz(inicio=None):
    """
    Devuelve la raíz del proyecto.

    Jupyter abre el kernel en la carpeta donde está guardado el cuaderno, no en la
    carpeta desde la que se lanzó JupyterLab. Como cada cuaderno vive en la carpeta
    de su fase (F1, F2, F3 o F4), si la carpeta de trabajo es una de ellas la raíz
    es la carpeta superior. Si el cuaderno se ejecuta desde la raíz, se usa tal cual.
    """
    inicio = Path.cwd() if inicio is None else Path(inicio)
    return inicio.parent if inicio.name in {"F1", "F2", "F3", "F4"} else inicio


RAIZ = localizar_raiz()

# Rutas de datos COMUNES a todas las fases: son exactamente las que usa el
# cuaderno de la Fase 2, de modo que un mismo archivo sirva a los dos.
DIR_CRUDO = RAIZ / "data" / "raw"          # datos originales: nunca se modifican
DIR_PROCESADO = RAIZ / "data" / "processed"  # salida del pipeline de la Fase 2
DIR_DOCS = RAIZ / "docs"                   # diccionario, fichas y metadatos
DIR_SRC = RAIZ / "src"                     # módulos propios reutilizables

# Una carpeta por fase para cuadernos e informes.
DIR_FASES = [RAIZ / f"F{n}" for n in (1, 2, 3, 4)]

for carpeta in [DIR_CRUDO, DIR_PROCESADO, DIR_DOCS, DIR_SRC, *DIR_FASES]:
    # parents=True crea las intermedias; exist_ok=True evita el error si ya existen,
    # que es lo que permite volver a ejecutar el cuaderno completo sin romperlo.
    carpeta.mkdir(parents=True, exist_ok=True)

print("Estructura creada bajo:", RAIZ.resolve())
# Se omiten las carpetas ocultas (.git, .venv, .ipynb_checkpoints) y las cachés.
for ruta in sorted(p.relative_to(RAIZ) for p in RAIZ.rglob("*") if p.is_dir()):
    if any(parte.startswith(".") or parte == "__pycache__" for parte in ruta.parts):
        continue
    print("  ", ruta.as_posix() + "/")

### 3.1 El archivo `.gitignore`

Quedan fuera del repositorio tres cosas: lo que se regenera (el entorno virtual, las
cachés), lo que es local de cada equipo (archivos del sistema operativo) y lo que es
secreto (credenciales). Se escribe **antes** del primer `git add .`: después ya es tarde,
porque el archivo quedó en el historial.

In [ ]:
CONTENIDO_GITIGNORE = """# Entorno virtual: se reconstruye con requirements.txt
.venv/

# Caché de Python
__pycache__/
*.pyc

# Jupyter
.ipynb_checkpoints/

# Windows
Thumbs.db
desktop.ini

# macOS
.DS_Store

# Variables de entorno y secretos
.env

# Datos pesados y modelos
*.ckpt
*.pth
"""

ARCHIVO_GITIGNORE = RAIZ / ".gitignore"
ARCHIVO_GITIGNORE.write_text(CONTENIDO_GITIGNORE, encoding="utf-8")

# Verificación de ida y vuelta: se relee lo escrito antes de darlo por hecho.
lineas = [l for l in ARCHIVO_GITIGNORE.read_text(encoding="utf-8").splitlines()
          if l and not l.startswith("#")]
assert ".venv/" in lineas, "El entorno virtual debe estar ignorado."
print(f"{ARCHIVO_GITIGNORE} escrito con {len(lineas)} reglas activas.")

### 3.2 El archivo `requirements.txt`

Declarar las dependencias es lo que separa «funciona en mi computador» de un entorno
reproducible. Se escriben con la versión exacta observada en el entorno, no con la última
que exista en el momento de instalar.

In [ ]:
def escribir_requirements(ruta, versiones):
    """
    Escribe requirements.txt a partir de las versiones observadas en el entorno.

    Parametros
    ----------
    ruta : Path
        Archivo de salida.
    versiones : dict
        Nombre de importacion -> version detectada (o None si falta).

    Retorna
    -------
    list of str
        Lineas escritas, para verificarlas.

    Lanza
    -----
    ValueError
        Si ninguna dependencia pudo detectarse.
    """
    lineas = []
    for nombre, version in versiones.items():
        if version is None:
            continue  # una librería ausente no se declara: primero se instala
        paquete = NOMBRE_EN_PYPI.get(nombre, nombre)
        lineas.append(f"{paquete}=={version}")

    if not lineas:
        raise ValueError("No se detectó ninguna dependencia instalada.")

    # Se agregan las herramientas del flujo de trabajo, que no se importan
    # desde el código pero sí forman parte del entorno reproducible.
    lineas += ["jupyterlab", "notebook", "ipykernel"]
    ruta.write_text("\n".join(sorted(lineas)) + "\n", encoding="utf-8")
    return sorted(lineas)


ARCHIVO_REQUIREMENTS = RAIZ / "requirements.txt"
requisitos = escribir_requirements(ARCHIVO_REQUIREMENTS, ENTORNO["versiones"])
print(ARCHIVO_REQUIREMENTS, "\n")
print("\n".join(requisitos))

## 4. Módulos: del cuaderno a `src/`

**Qué hace este paso.** Escribe un módulo `src/utilidades.py` y lo importa de vuelta.

**Por qué.** La Unidad 1 pide funciones, excepciones **y módulos**. Un cuaderno documenta y
comunica; un módulo es lo que permite que la misma función se use en las cuatro fases sin
copiarla. Aquí se mueve una función pequeña pero real: la que normaliza los nombres de las
columnas, que las fases siguientes usarán apenas carguen datos.

In [ ]:
CODIGO_MODULO = """\
\"\"\"Utilidades compartidas del proyecto MCDI500 - Grupo 9.

Este modulo lo genera el cuaderno de la Fase 1 y lo importan las fases siguientes.
\"\"\"

import re


class ProyectoError(Exception):
    \"\"\"Error propio del proyecto: permite distinguirlo de los de las librerias.\"\"\"


def normalizar_nombre(texto):
    \"\"\"Convierte un nombre de columna a minusculas, sin espacios ni signos.

    Ejemplo
    -------
    >>> normalizar_nombre('Residence Type ')
    'residence_type'
    \"\"\"
    if not isinstance(texto, str):
        raise ProyectoError(f\"Se esperaba texto y se recibio {type(texto).__name__}.\")
    limpio = texto.strip().lower()
    limpio = re.sub(r\"[^a-z0-9]+\", \"_\", limpio)   # todo lo que no sea letra o digito -> _
    return limpio.strip(\"_\")


def normalizar_columnas(nombres):
    \"\"\"Aplica normalizar_nombre a una lista y verifica que no se produzcan duplicados.\"\"\"
    normalizados = [normalizar_nombre(n) for n in nombres]
    if len(set(normalizados)) != len(normalizados):
        raise ProyectoError(\"La normalizacion produjo nombres duplicados.\")
    return normalizados
"""

ARCHIVO_MODULO = DIR_SRC / "utilidades.py"
ARCHIVO_MODULO.write_text(CODIGO_MODULO, encoding="utf-8")
print("Módulo escrito en:", ARCHIVO_MODULO)

Para importarlo hay que agregar `src/` a la lista de rutas donde Python busca módulos.
`invalidate_caches()` es necesario porque el archivo se acaba de crear: sin esa llamada,
Python puede no ver un módulo escrito después de arrancar el intérprete.